# Feature-selection comparison: recorded death

Compare logistic regression, decision tree, ANOVA, mutual information,
random-forest importance, RFE, Boruta, Lasso and Elastic Net logistic selection,
and XGBoost. Each method chooses **at most 20 original attributes**.

Run `python src/select_features.py` first. All methods use the same stratified
80/20 split and five validation folds within the training portion. Every fold
fits pruning, imputation, encoding, scaling, selection and the evaluation model
on its training rows only. Known identifiers and outcome dates/durations are
excluded. Mean validation **ROC AUC** is the primary comparison metric.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

project = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
runs = sorted((project / "results" / "feature_selection").glob("*/config.json"))
if not runs:
    raise FileNotFoundError("Run python src/select_features.py first.")
run = runs[-1].parent
config = json.loads((run / "config.json").read_text())
selected = json.loads((run / "selected_features.json").read_text())
rankings = pd.read_csv(run / "feature_rankings.csv")
print("Run:", run.name)
print("Methods:", ", ".join(selected))
print("Training patients:", config["train_rows"], "Test patients:", config["test_rows"])


Run: 20261101T134938_572565Z
Methods: logistic_regression, decision_tree, anova, mutual_information, random_forest, rfe, boruta, lasso, elastic_net, xgboost
Training patients: 314 Test patients: 79


## ROC AUC comparison

Order methods by mean validation ROC AUC. Fold standard deviation describes
variation across splits; it is not a confidence interval. Held-out test scores
are reporting only, not a tuning criterion. Baseline settings are fixed, not
Optuna-optimized.

ANOVA, mutual information and RFE use logistic regression for evaluation.
Other methods use their corresponding classifier; Boruta uses a random forest.
Scores therefore compare complete selection/classifier approaches, not the
selectors in isolation. Tree and coefficient scores should not be compared
directly with each other.


In [2]:
cv_summary = pd.read_csv(run / "cv_summary.csv").sort_values("roc_auc_mean", ascending=False)
test_results = pd.read_csv(run / "test_results.csv").set_index("method").loc[cv_summary["method"]].reset_index()
print("Primary metric: mean cross-validation ROC AUC")
display(cv_summary[["method", "roc_auc_mean", "roc_auc_std"]])
print("Held-out test ROC AUC")
display(test_results[["method", "n_features", "roc_auc"]])
print("Supporting metrics")
display(cv_summary)
display(test_results)


Primary metric: mean cross-validation ROC AUC
                method  roc_auc_mean  roc_auc_std
0                anova      0.777250     0.041036
1                lasso      0.772429     0.061488
2               boruta      0.768066     0.027436
3        random_forest      0.763827     0.021687
4   mutual_information      0.757560     0.048103
5          elastic_net      0.748080     0.051125
6              xgboost      0.736016     0.051270
7                  rfe      0.692009     0.057179
8        decision_tree      0.654580     0.069334
9  logistic_regression      0.589012     0.086578
Held-out test ROC AUC
                method  n_features   roc_auc
0                anova          20  0.742484
1                lasso          20  0.762092
2               boruta          20  0.781699
3        random_forest          20  0.764706
4   mutual_information          20  0.711765
5          elastic_net          20  0.751634
6              xgboost          20  0.700654
7                  rfe

## Five best attributes per method

Show only the **five highest-ranked chosen original attributes** for each
method. The complete selected lists (up to 20 per method) are saved together
in `top20_attributes_by_method.csv`: rows are ranks 1–20 and columns are methods.
Empty cells mean that a method chose fewer attributes.

Rank scores have different meanings and scales between methods. They should
not be compared numerically across methods. Zero-score attributes are not added
to fill a list. One-hot categories are grouped under their original names.


In [3]:
import sys
if str(project) not in sys.path:
    sys.path.insert(0, str(project))
from src.feature_selection.reporting import save_selection_reports

all_selected, attribute_frequency = save_selection_reports(rankings, run)
for method in selected:
    print(method)
    best_five = rankings.loc[
        (rankings["method"] == method) & rankings["selected"],
        ["rank", "feature", "importance"],
    ].sort_values("rank").head(5)
    display(best_five.reset_index(drop=True))
print("All selected attributes saved to:", run / "top20_attributes_by_method.csv")


logistic_regression
   rank            feature  importance
0     1          Wskazania    1.113817
1     2    typo urządzenia    0.981071
2     3           Es'_mean    0.967733
3     4          producent    0.956587
4     5  TL_cca,CCA,PI-CCA    0.906851
decision_tree
   rank                  feature  importance
0     1            B_NT - proBNP    0.303318
1     2                 MDRDcorr    0.081884
2     3  TR_ica,ICA,PSV-ICA cm/s    0.070881
3     4           TEB,0,PEP mean    0.064213
4     5                 Ea'_mean    0.052663
anova
   rank            feature  importance
0     1      B_NT - proBNP   39.312751
1     2        E_diastarea   36.348372
2     3           Ea'_mean   35.288595
3     4               LVEF   31.088943
4     5  E_LV2j   rozk cm2   29.844524
mutual_information
   rank               feature  importance
0     1         B_NT - proBNP    0.112879
1     2  6mwt_,rr_,RR L,0 SBP    0.105518
2     3                  LVEF    0.088561
3     4           beta_bloker    0.

## Boruta: confirmed relevant attributes

Confirmed relevance is different from being included in the capped comparison.
Boruta keeps its confirmed/tentative/rejected labels. We call an original
attribute confirmed if any encoded category is confirmed, otherwise tentative
if any category is tentative. This aggregation is a reporting convention, not
a joint statistical test. The capped list is ordered by Boruta rank, summed
median importance and name, so it may include tentative or rejected attributes.
Counts are calculated from the saved final training fit and may change by run.


In [4]:
boruta_rankings = rankings.loc[rankings["method"] == "boruta"].copy()
if boruta_rankings.empty:
    print("This run has no Boruta results.")
else:
    confirmed = boruta_rankings.loc[boruta_rankings["boruta_status"] == "confirmed"]
    print(f"BORUTA CONFIRMED {len(confirmed)} RELEVANT ORIGINAL ATTRIBUTES")
    display(boruta_rankings["boruta_status"].value_counts().reindex(
        ["confirmed", "tentative", "rejected"], fill_value=0
    ).rename_axis("status").to_frame("attributes"))
    print("Status of attributes used in the capped comparison:")
    display(boruta_rankings.loc[boruta_rankings["selected"], "boruta_status"].value_counts())
    print("Confirmed relevant names are available in feature_rankings.csv (boruta_status=confirmed).")


BORUTA CONFIRMED 17 RELEVANT ORIGINAL ATTRIBUTES
           attributes
status               
confirmed          17
tentative           8
rejected          262
Status of attributes used in the capped comparison:
boruta_status
confirmed    17
tentative     3
Name: count, dtype: int64
Confirmed relevant names are available in feature_rankings.csv (boruta_status=confirmed).


## How often was each attribute chosen?

Count how many methods included each attribute in their **complete final lists
of up to 20**, not just the five shown above. Each method contributes at most
one appearance per attribute. The percentage uses the number of methods in
this run. Attributes chosen by no methods are omitted.

This is agreement between methods, not proof of relevance or causality.
Cross-validation selection frequency is a separate measure and is saved to
`fold_selection_frequency.csv` without printing another large table here.


In [5]:
with pd.option_context("display.max_rows", None):
    display(attribute_frequency)
print("Counts saved to:", run / "attribute_selection_frequency.csv")

folds = pd.read_csv(run / "fold_selected_features.csv")
fold_frequency = folds.groupby(["method", "feature"]).size().rename("folds_selected").reset_index()
fold_frequency = fold_frequency.sort_values(
    ["method", "folds_selected", "feature"], ascending=[True, False, True]
)
fold_frequency.to_csv(run / "fold_selection_frequency.csv", index=False)


                           feature  methods_count  methods_percent
0                    B_NT - proBNP             10            100.0
1                     B_Kreatynina              8             80.0
2                    6mwt_,Dystans              7             70.0
3                      E_diastarea              7             70.0
4                             LVEF              7             70.0
5                          Ant_SBP              5             50.0
6                 E_skurcz,LAA cm2              5             50.0
7                         Ea'_mean              5             50.0
8                    B_GFR wg MDRD              4             40.0
9                     B_Lbazocytów              4             40.0
10                       Chadsvasc              4             40.0
11               E_LV2j   rozk cm2              4             40.0
12              E_Lewy przedsionek              4             40.0
13                  E_rozk,RAA cm2              4             

## Settings and interpretation

The target assumes `1=recorded death`, `0=no recorded death`; verify against the
data dictionary. The results describe associations, not causes of death or
fixed-horizon mortality risk. Predictor collection timing is incompletely
known, so further outcome proxies may need exclusion. Preprocessing uses
training-fitted median/mode imputation, not full-dataset MissForest estimates.

The original `test_feature_selection_techniques.ipynb` is preserved as an
exploratory notebook. Its whole-dataset rankings are not validation scores.
Earlier standalone Lasso/Elastic Net/XGBoost evaluations used all patients in
five-fold validation; these unified results use the 80% training portion and
are not numerically interchangeable with those earlier scores.


In [6]:
display(config)


{'excluded_columns': ['Data przyjęcia', 'cid', 'cid.1', 'data implantacji', 'data zgonu lub konca obserwacji', 'day_count_begin_to_end', 'tryb', 'zgon'], 'missing_target_rows_removed': 0, 'target': 'zgon', 'target_coding': {'0': 'no recorded death', '1': 'recorded death'}, 'scope': 'Exploratory association with recorded death, not causal attribution or fixed-horizon risk.', 'n_features': 20, 'random_state': 42, 'primary_metric': 'roc_auc', 'comparison_rule': 'Higher mean cross-validation ROC AUC; test scores are reporting only.', 'test_fraction': 0.2, 'cv_folds': 5, 'train_rows': 314, 'test_rows': 79, 'class_counts': {'1': 226, '0': 167}, 'source': '/home/anka335/Desktop/studia/sem5/Problem Classes I/u-smile_problem_classes/data/raw/cardiac_patient_data.csv', 'source_sha256': '69a99872287b2b05e08d860c28b44e090d3d2deb6e3ba4b76befadf4cdcc4a30', 'versions': {'pandas': '3.0.6', 'numpy': '2.5.4', 'sklearn': '1.9.1', 'boruta': '0.4.3', 'xgboost': '3.4.1'}, 'models': {'logistic_regression': {